In [2]:
# 导入必要的库（就像是准备食材）
import torch
from torch import nn
from transformers import PretrainedConfig

# 定义模型配置类：决定你的机器人有多聪明（大）
class ModelConfig(PretrainedConfig):
    model_type = "My-Tiny-LLaMA"
    def __init__(
        self,
        dim: int = 256,      # 机器人的脑细胞维度（越大学习能力越强，但也越占内存）
        n_layers: int = 8,    # 脑细胞的层数
        n_heads: int = 8,     # 机器人的“注意力”头数
        vocab_size: int = 3200, # 机器人能认识多少个词
        max_seq_len: int = 512, # 机器人一次最长能读多长的句子
        **kwargs
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

# 实例化一份配置
config = ModelConfig()
print("配置创建成功！我们的模型维度是：", config.dim)

c:\Users\bzylewis\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


配置创建成功！我们的模型维度是： 256


In [22]:
import sys
!{sys.executable} -m pip install torch transformers -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
     - ------------------------------------- 5.2/113.8 MB 27.1 MB/s eta 0:00:05
     ---- --------------------------------- 12.6/113.8 MB 31.2 MB/s eta 0:00:04
     ----- -------------------------------- 17.3/113.8 MB 28.8 MB/s eta 0:00:04
     -------- ----------------------------- 25.4/113.8 MB 31.2 MB/s eta 0:00:03
     ----------- -------------------------- 33.3/113.8 MB 32.6 MB/s eta 0:00:03
     ------------- ------------------------ 41.2/113.8 MB 33.5 MB/s eta 0:00:03
     ---------------- --------------------- 49.3/113.8 MB 34.2 MB/s eta 0:00:02
     ------------------- ------------------ 56.9/113.8 MB 34.2 MB/s eta 0:00:02
     --------------------- ---------------- 65.3/113.8 MB 34.8 MB/s eta 0:00:02
     ------------------------ ------------- 72.4/113.8 MB 34.7 MB/s eta 0:00:02
     -------------------------- ----------- 80.7/113.8 MB 35.0 MB/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cli-base 0.6.0 requires click<8.2, but you have click 8.3.1 which is incompatible.


In [3]:
from transformers import PretrainedConfig

class ModelConfig(PretrainedConfig):
    model_type = "Tiny-LLaMA"
    def __init__(
        self,
        dim: int = 512,      # 脑细胞（向量）的维度
        n_layers: int = 8,    # 模型的层数
        n_heads: int = 8,     # 注意力头的数量
        n_kv_heads: int = 4,  # 为了节省内存，KV头数可以少一点
        vocab_size: int = 3200, # 认得多少个词
        hidden_dim: int = 1024, # 思考层的中间维度
        norm_eps: float = 1e-5,
        max_seq_len: int = 512, # 一次最多读多长的句子
        **kwargs
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.norm_eps = norm_eps
        self.max_seq_len = max_seq_len

# 创建一个配置实例
config = ModelConfig()
print(f"成功创建模型配置！模型维度: {config.dim}, 层数: {config.n_layers}")

成功创建模型配置！模型维度: 512, 层数: 8


In [4]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim)) # 可学习的缩放参数

    def _norm(self, x):
        # 核心数学公式：除以均方根
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return self.weight * self._norm(x.float()).type_as(x)

print("RMSNorm 稳压器组件完成！")

RMSNorm 稳压器组件完成！


In [5]:
import torch.nn.functional as F

class FeedForward(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        # LLaMA2 的特色：三路线性变换
        self.w1 = nn.Linear(config.dim, config.hidden_dim, bias=False)
        self.w2 = nn.Linear(config.hidden_dim, config.dim, bias=False)
        self.w3 = nn.Linear(config.dim, config.hidden_dim, bias=False)

    def forward(self, x):
        # SwiGLU 激活函数： silu(w1(x)) * w3(x) 然后再过 w2
        return self.w2(F.silu(self.search_w1(x)) * self.w3(x))

# 修正代码中的小错误并运行：
class FeedForward(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.w1 = nn.Linear(config.dim, config.hidden_dim, bias=False)
        self.w2 = nn.Linear(config.hidden_dim, config.dim, bias=False)
        self.w3 = nn.Linear(config.dim, config.hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

print("MLP 处理器组件完成！")

MLP 处理器组件完成！


In [6]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    # 预计算旋转频率矩阵
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # 复数形式
    return freqs_cis

# 试试预计算一下
test_freqs = precompute_freqs_cis(64, 512)
print("RoPE 频率矩阵形状:", test_freqs.shape)

RoPE 频率矩阵形状: torch.Size([512, 32])


In [17]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """将 KV 的头数重复 n_rep 次，以匹配 Query 的头数"""
    if n_rep == 1:
        return x
    bs, slen, n_kv_heads, head_dim = x.shape
    return (
        x[:, :, :, None, :]
        .expand(bs, slen, n_kv_heads, n_rep, head_dim)
        .reshape(bs, slen, n_kv_heads * n_rep, head_dim)
    )

class Attention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep = self.n_heads // self.n_kv_heads # 计算需要重复几次
        self.head_dim = config.dim // config.n_heads
        
        self.wq = nn.Linear(config.dim, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.dim, bias=False)

    def forward(self, x, freqs_cis):
        batch, seqlen, _ = x.shape
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        xq = xq.view(batch, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(batch, seqlen, self.n_kv_heads, self.head_dim)
        xv = xv.view(batch, seqlen, self.n_kv_heads, self.head_dim)

        # 应用 RoPE 旋转位置编码 (假设你之前定义的 apply_rotary_emb 还在)
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis)

        # --- 修正处：把 KV 重复次数补齐到和 Q 一样多 ---
        xk = repeat_kv(xk, self.n_rep) # 4变成8
        xv = repeat_kv(xv, self.n_rep) # 4变成8

        # 现在的 xq 是 (bs, seq, 8, dim), xk 也是 (bs, seq, 8, dim)
        # 接下来计算就不会报错了
        scores = torch.matmul(xq.transpose(1, 2), xk.transpose(1, 2).transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        
        output = torch.matmul(scores, xv.transpose(1, 2)) 
        output = output.transpose(1, 2).contiguous().view(batch, seqlen, -1)
        
        return self.wo(output)

print("修正后的 Attention 模块已就绪！")

修正后的 Attention 模块已就绪！


In [18]:
class TransformerBlock(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.attention = Attention(config)
        self.feed_forward = FeedForward(config)
        self.attention_norm = RMSNorm(config.dim, eps=config.norm_eps)
        self.ffn_norm = RMSNorm(config.dim, eps=config.norm_eps)

    def forward(self, x, freqs_cis):
        # 第一步：先稳压，再看（Attention）
        # 这里用到了“残差连接” (x + ...)，防止模型学着学着把前面的忘了
        x = x + self.attention(self.attention_norm(x), freqs_cis)
        # 第二步：再稳压，再思考（MLP）
        x = x + self.feed_forward(self.ffn_norm(x))
        return x

print("TransformerBlock 思考层组装完毕！")

TransformerBlock 思考层组装完毕！


In [19]:
class Llama2(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        # 词嵌入层：把词语 ID 变成 512 维的向量
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.dim)
        # 堆叠 8 层思考层
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.dim, eps=config.norm_eps)
        # 输出层：把向量变回词语的概率
        self.output = nn.Linear(config.dim, config.vocab_size, bias=False)
        # 预计算指南针频率
        self.freqs_cis = precompute_freqs_cis(config.dim // config.n_heads, config.max_seq_len)

    def forward(self, tokens):
        _batch, seqlen = tokens.shape
        h = self.tok_embeddings(tokens)
        # 取出对应长度的指南针频率
        freqs_cis = self.freqs_cis[:seqlen].to(h.device)

        for layer in self.layers:
            h = layer(h, freqs_cis)

        h = self.norm(h)
        return self.output(h)

# 实例化最终模型
model = Llama2(config)
print(f"恭喜！你的 LLaMA2 模型已成功合体！总参数量约为: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

恭喜！你的 LLaMA2 模型已成功合体！总参数量约为: 22.16 M


In [20]:
# 模拟一些语料数据
corpus = [
    "我爱学习人工智能",
    "大模型技术正在改变世界",
    "今天天气真不错",
    "我想亲手做一个大模型"
]

# 将语料保存到本地文件
with open("corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

In [22]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. 初始化一个空的 BPE 模型
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace() # 基础分词规则：按空格/标点切分

# 2. 设置训练器，定义特殊符号
trainer = BpeTrainer(
    vocab_size=3200, # 词表大小，小白建议先设小一点
    special_tokens=["<s>", "</s>", "<unk>", "<pad>", "<mask>"]
)

# 3. 开始训练
files = ["corpus.txt"]
tokenizer.train(files, trainer)

# 4. 保存字典
tokenizer.save("tokenizer.json")
print("分词器字典训练并保存成功！")

分词器字典训练并保存成功！


In [23]:
# 加载刚刚训练好的分词器
tokenizer = Tokenizer.from_file("tokenizer.json")

# 测试翻译一句话
test_sentence = "我爱学习"
encoded = tokenizer.encode(test_sentence)

print(f"原始句子: {test_sentence}")
print(f"转换后的 ID 序列: {encoded.ids}")
print(f"回译文字: {encoded.tokens}")

原始句子: 我爱学习
转换后的 ID 序列: [51, 19, 9]
回译文字: ['我爱', '学', '习']


In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length
        with open(file_path, 'r', encoding='utf-8') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        # 将一行文本转换成数字 ID
        text = self.lines[idx]
        encoding = self.tokenizer.encode(text)
        ids = encoding.ids
        
        # 补齐或截断到固定长度（模型需要整齐的数据）
        if len(ids) > self.max_length:
            ids = ids[:self.max_length]
        else:
            ids = ids + [0] * (self.max_length - len(ids)) # 0 通常是 <pad>
            
        # 训练大模型的目标是：给它前 n 个字，预测第 n+1 个字
        # 所以输入 X 是 [0 到 n-1]，标签 Y 是 [1 到 n]
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y

# 准备数据加载器
dataset = TextDataset("corpus.txt", tokenizer, max_length=config.max_seq_len)
train_loader = DataLoader(dataset, batch_size=2, shuffle=True)

print("数据加载器准备完毕！")

数据加载器准备完毕！


In [26]:
import torch.optim as optim

# 设置学习目标：交叉熵损失（用于预测下一个词的概率）
criterion = nn.CrossEntropyLoss()

# 设置学习工具：Adam 优化器（它会自动调整模型参数）
optimizer = optim.AdamW(model.parameters(), lr=5e-4)

print("优化器和损失函数已就绪。")

优化器和损失函数已就绪。


In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"正在使用 {device} 进行训练...")

model.train() # 切换到训练模式
for epoch in range(10): # 学习 10 遍
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        # 1. 机器人猜一猜
        logits = model(x)
        
        # 2. 算算错得有多离谱
        # logits 形状是 [batch, seq, vocab], y 是 [batch, seq]
        loss = criterion(logits.view(-1, config.vocab_size), y.view(-1))
        
        # 3. 纠错过程（反向传播）
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f"第 {epoch+1} 遍学习完成，误差(Loss): {loss.item():.4f}")

print("小规模训练测试完成！")

正在使用 cpu 进行训练...
第 1 遍学习完成，误差(Loss): 0.2919
第 2 遍学习完成，误差(Loss): 0.0127
第 3 遍学习完成，误差(Loss): 0.0080
第 4 遍学习完成，误差(Loss): 0.0070
第 5 遍学习完成，误差(Loss): 0.0062
第 6 遍学习完成，误差(Loss): 0.0054
第 7 遍学习完成，误差(Loss): 0.0046
第 8 遍学习完成，误差(Loss): 0.0039
第 9 遍学习完成，误差(Loss): 0.0033
第 10 遍学习完成，误差(Loss): 0.0029
小规模训练测试完成！


In [28]:
@torch.no_grad() # 推理时不需要计算梯度，节省显存
def generate(model, tokenizer, prompt, max_new_tokens=20, temperature=0.7):
    model.eval() # 切换到评价模式
    
    # 1. 把你的话转换成 ID
    tokens = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)
    
    for _ in range(max_new_tokens):
        # 2. 扔给模型看
        logits = model(tokens)
        
        # 3. 只看最后一个词的预测结果，并应用温度系数（Temperature）
        # 温度越高，说话越随机；温度越低，说话越保守
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        
        # 4. 从概率中抽样一个词 ID
        next_token = torch.multinomial(probs, num_samples=1)
        
        # 5. 把新词拼接到原来的句子后面
        tokens = torch.cat((tokens, next_token), dim=1)
        
        # 如果抽到了停止符，就提前结束
        if next_token.item() == tokenizer.token_to_id("</s>"):
            break
            
    # 6. 把这一串 ID 变回人类看得懂的文字
    return tokenizer.decode(tokens[0].tolist())

print("推理函数已就绪！")

推理函数已就绪！


In [29]:
prompt = "我爱"
answer = generate(model, tokenizer, prompt, max_new_tokens=10)
print(f"输入: {prompt}")
print(f"模型输出: {answer}")

输入: 我爱
模型输出: 我爱


In [30]:
# 保存模型参数（权重）
torch.save(model.state_dict(), "my_tiny_llama.pth")
print("你的模型‘进度’已保存到：my_tiny_llama.pth")

你的模型‘进度’已保存到：my_tiny_llama.pth


In [31]:
# 1. 先准备一个一模一样的空壳模型
new_config = ModelConfig() # 确保配置参数和作者的一致
new_model = Llama2(new_config).to(device)

# 2. 把保存的权重‘填’进去
state_dict = torch.load("my_tiny_llama.pth", map_location=device)
new_model.load_state_dict(state_dict)

print("模型加载成功！它现在拥有了之前保存的所有知识。")

模型加载成功！它现在拥有了之前保存的所有知识。


In [32]:
# 使用刚刚加载的模型进行推理
test_prompt = "大模型技术"
result = generate(new_model, tokenizer, test_prompt, max_new_tokens=15)
print(f"验证加载后的输出: {result}")

验证加载后的输出: 大模型技术 真不错
